# Training and evaluating a vanilla neural network

In [22]:
import json
from collections import Counter
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, precision_recall_fscore_support
import joblib
from sentence_transformers import SentenceTransformer
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm

data_path = "../data"

with open(f"{data_path}/train_nn_set.json", "r") as f:
    train_nn_set = json.load(f)

with open(f"{data_path}/validate_set.json", "r") as f:
    validate_set = json.load(f)

with open(f"{data_path}/test_set.json", "r") as f:
    test_set = json.load(f)

VANILLA_NN_MODEL_PATH = "../models/vanilla_nn/3H1L_vanilla_nn_model.pkl"
SENTENCE_EMBED_TRAIN_VECTOR_PATH = "../models/vanilla_nn/3H1L_train_nn_sentence_embeddings.pkl"
SENTENCE_EMBED_VALIDATE_VECTOR_PATH = "../models/vanilla_nn/3H1L_validate_nn_sentence_embeddings.pkl"
SENTENCE_EMBED_TEST_VECTOR_PATH = "../models/vanilla_nn/3H1L_test_nn_sentence_embeddings.pkl"


### Transform data into appropriate format

Create Python list of dicts

In [23]:
def flatten_transform(target_set):

    modified_set = []

    for item in target_set:
        date = item["date"]
        answer = item["answer"]
        for index, headline in enumerate(item["headlines"]):
            if index == answer:
                fake = 1
            else:
                fake = 0
            entry = {
                "Date": date,
                "Headline": headline,
                "Fake": fake
            }
            modified_set.append(entry)

    return modified_set

train_set_nn_ready = flatten_transform(train_nn_set)
validate_set_nn_ready = flatten_transform(validate_set)
test_set_nn_ready = flatten_transform(test_set)


Transform into dataframe and create X and y classes

In [24]:
train_nn_df = pd.DataFrame(train_set_nn_ready)
X_train_nn_text = train_nn_df["Headline"]
y_train = train_nn_df["Fake"]

validate_nn_df = pd.DataFrame(validate_set_nn_ready)
X_validate_nn_text = validate_nn_df["Headline"]
y_validate = validate_nn_df["Fake"]

test_df = pd.DataFrame(test_set_nn_ready)
X_test_nn_text = test_df["Headline"]
y_test = test_df["Fake"] # May not actually use this given quiz level accuracy is primary metric

### Vanilla neural network

Get sentence embeddings

In [25]:
try:
    X_train_nn_SE = joblib.load(SENTENCE_EMBED_TRAIN_VECTOR_PATH)
    X_validate_nn_SE = joblib.load(SENTENCE_EMBED_VALIDATE_VECTOR_PATH)
    X_test_nn_SE = joblib.load(SENTENCE_EMBED_TEST_VECTOR_PATH)
    print("Loaded model and embeddings from disk.")

except FileNotFoundError:

    print("No saved embeddings found, creating new ones...")

    # Create embedding model
    model = SentenceTransformer("all-MiniLM-L6-v2")

    # Embed both train and test sets
    X_train_nn_SE = model.encode(list(X_train_nn_text))
    X_validate_nn_SE = model.encode(list(X_validate_nn_text))
    X_test_nn_SE = model.encode(list(X_test_nn_text))

    joblib.dump(X_train_nn_SE, SENTENCE_EMBED_TRAIN_VECTOR_PATH)
    joblib.dump(X_validate_nn_SE, SENTENCE_EMBED_VALIDATE_VECTOR_PATH)
    joblib.dump(X_test_nn_SE, SENTENCE_EMBED_TEST_VECTOR_PATH)
    print("Model and embeddings saved.")

print("Embeddings matrix shape (train):", X_train_nn_SE.shape)
print("Embeddings matrix shape (validate):", X_validate_nn_SE.shape)
print("Embeddings matrix shape (test):", X_test_nn_SE.shape)

Loaded model and embeddings from disk.
Embeddings matrix shape (train): (812, 384)
Embeddings matrix shape (validate): (268, 384)
Embeddings matrix shape (test): (268, 384)


Convert to PyTorch tensors and package into batches

In [26]:
X_train_tensor = torch.FloatTensor(X_train_nn_SE)
y_train_tensor = torch.FloatTensor(y_train.to_numpy()).unsqueeze(1)
X_validate_tensor = torch.FloatTensor(X_validate_nn_SE)
y_validate_tensor = torch.FloatTensor(y_validate.to_numpy()).unsqueeze(1)
X_test_tensor = torch.FloatTensor(X_test_nn_SE)
y_test_tensor = torch.FloatTensor(y_test.to_numpy()).unsqueeze(1)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

C:\Users\popro\AppData\Local\Temp\ipykernel_21284\722502672.py:2: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  y_train_tensor = torch.FloatTensor(y_train.to_numpy()).unsqueeze(1)


Define neural network

In [27]:
class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super(NeuralNetwork, self).__init__()
        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        output1 = self.relu(self.layer1(x))
        output2 = self.relu(self.layer2(output1))
        output3 = self.layer3(output2)
        return output3

Instantiate the model

In [28]:
input_size = X_train_tensor.shape[1]
model = NeuralNetwork(input_size)

Define loss function

In [29]:
loss_function = nn.BCEWithLogitsLoss()
optimiser = optim.Adam(model.parameters(), lr=0.001)

Training loop

In [32]:
EPOCHS = 20

for epoch in range(EPOCHS):
    model.train()
    for batch_X, batch_y in train_loader:
        optimiser.zero_grad()
        outputs = model(batch_X)
        loss = loss_function(outputs, batch_y)
        loss.backward()
        optimiser.step()

    model.eval()
    with torch.no_grad():
        val_outputs = model(X_validate_tensor)
        val_loss = loss_function(val_outputs, y_validate_tensor)

    print(f'Epoch [{epoch+1}/{EPOCHS}], Train Loss: {loss.item():.3f}, Val Loss: {val_loss.item():.3f}')

Epoch [1/20], Train Loss: 0.554, Val Loss: 0.544
Epoch [2/20], Train Loss: 0.718, Val Loss: 0.525
Epoch [3/20], Train Loss: 0.474, Val Loss: 0.527
Epoch [4/20], Train Loss: 0.350, Val Loss: 0.527
Epoch [5/20], Train Loss: 0.135, Val Loss: 0.537
Epoch [6/20], Train Loss: 0.275, Val Loss: 0.561
Epoch [7/20], Train Loss: 0.331, Val Loss: 0.585
Epoch [8/20], Train Loss: 0.228, Val Loss: 0.614
Epoch [9/20], Train Loss: 0.153, Val Loss: 0.652
Epoch [10/20], Train Loss: 0.098, Val Loss: 0.691
Epoch [11/20], Train Loss: 0.033, Val Loss: 0.772
Epoch [12/20], Train Loss: 0.045, Val Loss: 0.796
Epoch [13/20], Train Loss: 0.035, Val Loss: 0.857
Epoch [14/20], Train Loss: 0.023, Val Loss: 0.894
Epoch [15/20], Train Loss: 0.020, Val Loss: 0.934
Epoch [16/20], Train Loss: 0.009, Val Loss: 0.995
Epoch [17/20], Train Loss: 0.012, Val Loss: 1.032
Epoch [18/20], Train Loss: 0.013, Val Loss: 1.069
Epoch [19/20], Train Loss: 0.004, Val Loss: 1.110
Epoch [20/20], Train Loss: 0.005, Val Loss: 1.138
